# Step 5: Question generation with Mistral (Ollama)

For every gold chunk (`data/gold/gold_chunks.jsonl`, step 4) this notebook asks a local
Mistral model to write **one question** that

* is written in the **same language as the chunk** (German chunks → German question, English → English),
* can be answered **only from that chunk**,
* uses **wording close to the chunk's own wording**,

together with a short answer copied word for word from the chunk.

Every generated question is checked automatically. If a check fails, the model is asked again
(with the reason), up to `MAX_ATTEMPTS` times:

| Check | Why |
|---|---|
| answer is copied word for word from the chunk | the answer is really in the chunk |
| answer is not already written in the question | the question does not give the answer away |
| most of the question's content words appear in the chunk | wording similar to the chunk |
| a keyword search (BM25) over **all** chunks of that language ranks the gold chunk first | the question points to this chunk and no other |
| the question is in the chunk's language | no language mix-ups |

**Output**

* `data/questions/questions.jsonl` – accepted questions
* `data/questions/rejected.jsonl` – chunks where no attempt passed, with the last attempt and the reasons

## Before you start

1. Install Ollama: https://ollama.com/download
2. Download the model (in a terminal):
   ```
   ollama pull mistral
   ```
3. Make sure Ollama is running (the desktop app, or `ollama serve` in a terminal).
4. Install the Python packages used here:
   ```
   pip install requests pandas
   ```
5. Start Jupyter from the repository folder and open this notebook from `notebooks/`.

## 1. Settings

In [ ]:
import json
import re
import sys
import time
from pathlib import Path

import pandas as pd
import requests

# The notebook lives in notebooks/, the repository root is one level up.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "ingest"))
from lexical import BM25, tokenize  # same tokenizer/stopwords as the gold-chunk selection

GOLD_CHUNKS = ROOT / "data" / "gold" / "gold_chunks.jsonl"
ALL_CHUNKS = ROOT / "data" / "chunks" / "chunks.jsonl"
OUT_DIR = ROOT / "data" / "questions"

OLLAMA_URL = "http://localhost:11434/api/chat"
MODEL = "mistral"        # any model you have pulled with `ollama pull ...`
TEMPERATURE = 0.2        # low = closer to the chunk's wording, more repeatable
SEED = 42                # fixed seed so a re-run gives the same questions
MAX_ATTEMPTS = 5         # tries per chunk before it is marked as rejected

# Thresholds for the automatic checks
MIN_WORD_OVERLAP = 0.7   # share of the question's content words that must appear in the chunk
REQUIRED_RANK = 1        # the gold chunk must be the top BM25 hit for the question

LANGUAGE_NAMES = {"DE": "German", "EN": "English"}

print("Repository root:", ROOT)

## 2. Check that Ollama is running and the model is available

In [ ]:
try:
    tags = requests.get("http://localhost:11434/api/tags", timeout=5).json()
except requests.ConnectionError:
    raise SystemExit("Ollama is not running. Start the Ollama app or run `ollama serve`.")

installed = [m["name"] for m in tags.get("models", [])]
print("Installed models:", installed)
assert any(name.split(":")[0] == MODEL.split(":")[0] for name in installed), \
    f"Model '{MODEL}' not found. Run: ollama pull {MODEL}"

## 3. Load the gold chunks and build the search index

The BM25 index covers **all** chunks of each language (not only the gold ones), so the
uniqueness check is done against everything a retriever could find.

In [ ]:
def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f]

gold_chunks = read_jsonl(GOLD_CHUNKS)
all_chunks = read_jsonl(ALL_CHUNKS)

# One BM25 index per language
index = {}
for lang in LANGUAGE_NAMES:
    lang_chunks = [c for c in all_chunks if c["language"] == lang]
    index[lang] = {
        "ids": [c["chunk_id"] for c in lang_chunks],
        "bm25": BM25([tokenize(c["text"], lang) for c in lang_chunks]),
    }

print(pd.Series([c["language"] for c in gold_chunks]).value_counts().rename("gold chunks"))

## 4. The prompt

The prompt is in English for both languages, but tells the model which language to write in.
The model must answer in JSON, so the output can be read reliably.

In [ ]:
SYSTEM_PROMPT = """You write evaluation questions for a retrieval test.
You are given one text passage. Write exactly one question about a specific fact in the passage.

Rules:
1. Write the question in {language}. The passage is in {language}.
2. The question must be answerable from this passage alone, and it must be about a detail
   (a name, number, date, place, object or reason) that is specific to this passage.
3. Reuse the passage's own words and phrases in the question. Do not paraphrase or use synonyms.
4. Do not put the answer in the question.
5. The answer must be short (1 to 12 words) and copied exactly, character for character, from the passage.
6. Do not ask about figure numbers, page numbers or literature references.

Reply only with JSON in this form:
{{"question": "...", "answer": "..."}}"""

USER_PROMPT = """Passage:
\"\"\"
{text}
\"\"\""""

# Sent back to the model when a check fails, so the next attempt can fix it
RETRY_PROMPT = """Your question did not pass these checks:
{problems}
Write a new question and answer for the same passage, following all rules. Reply only with JSON."""

## 5. Calling the model

In [ ]:
RESPONSE_SCHEMA = {
    "type": "object",
    "properties": {"question": {"type": "string"}, "answer": {"type": "string"}},
    "required": ["question", "answer"],
}

def ask_model(messages, attempt):
    """Send the conversation to Ollama and return the parsed JSON reply."""
    response = requests.post(OLLAMA_URL, timeout=300, json={
        "model": MODEL,
        "messages": messages,
        "stream": False,
        "format": RESPONSE_SCHEMA,  # Ollama forces the reply into this JSON shape
        "options": {"temperature": TEMPERATURE, "seed": SEED + attempt},
    })
    response.raise_for_status()
    content = response.json()["message"]["content"]
    return content, json.loads(content)

## 6. The automatic checks

In [ ]:
def answer_pattern(answer):
    """Regex matching the answer's words in order, ignoring case and the punctuation/quotes between them."""
    words = re.findall(r"\w+", answer)
    if not words:
        return None
    return re.compile(r"\b" + r"\W+".join(map(re.escape, words)) + r"\b", re.IGNORECASE)

def find_in_chunk(answer, chunk_text):
    """Return the exact span of the chunk that matches the answer, or None."""
    pattern = answer_pattern(answer)
    match = pattern.search(chunk_text) if pattern else None
    return match.group(0) if match else None

def looks_like_language(text, lang):
    """Rough language check: count common German vs English function words."""
    words = re.findall(r"\w+", text.lower())
    de = sum(w in {"der", "die", "das", "und", "wer", "was", "wann", "wo", "welche", "welcher",
                   "welches", "wie", "im", "von", "mit", "den", "dem", "des", "wurde", "ist", "zu"} for w in words)
    en = sum(w in {"the", "and", "who", "what", "when", "where", "which", "how", "in", "of", "with",
                   "was", "were", "is", "to", "did", "does"} for w in words)
    return de >= en if lang == "DE" else en >= de

def check(item, chunk):
    """Run all checks. Returns (problems, details); an empty problem list means accepted."""
    lang, text = chunk["language"], chunk["text"]
    question, answer = item.get("question", "").strip(), item.get("answer", "").strip()
    problems, details = [], {}

    if not question.endswith("?"):
        problems.append("The question must be a single question ending with '?'.")

    span = find_in_chunk(answer, text)
    details["answer_span"] = span
    if span is None:
        problems.append("The answer is not copied exactly from the passage.")
    elif answer_pattern(span).search(question):
        problems.append("The question already contains the answer.")
    elif len(span.split()) > 12:
        problems.append("The answer is longer than 12 words.")

    q_tokens = tokenize(question, lang)
    chunk_tokens = set(tokenize(text, lang))
    overlap = sum(t in chunk_tokens for t in q_tokens) / max(len(q_tokens), 1)
    details["word_overlap"] = round(overlap, 2)
    if overlap < MIN_WORD_OVERLAP:
        problems.append(f"Only {overlap:.0%} of the question's content words are taken from the passage; "
                        "reuse the passage's own words.")

    scores = index[lang]["bm25"].scores(q_tokens)
    ranking = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    rank = [index[lang]["ids"][i] for i in ranking].index(chunk["chunk_id"]) + 1
    details["gold_rank"] = rank
    if rank > REQUIRED_RANK:
        problems.append("The question also fits other passages; ask about a detail that is specific to this passage.")

    if not looks_like_language(question, lang):
        problems.append(f"The question must be written in {LANGUAGE_NAMES[lang]}.")

    return problems, details

## 7. Generate the questions

This loops over all gold chunks. With Mistral 7B on a laptop, expect roughly 5–20 seconds per attempt.
Results are written to disk after every chunk, so you can stop and re-run the cell: chunks that are
already done are skipped.

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
questions_path = OUT_DIR / "questions.jsonl"
rejected_path = OUT_DIR / "rejected.jsonl"

# Resume support: skip chunks that already have a result
done = set()
for path in (questions_path, rejected_path):
    if path.exists():
        done.update(r["chunk_id"] for r in read_jsonl(path))

for n, chunk in enumerate(gold_chunks, start=1):
    if chunk["chunk_id"] in done:
        continue
    lang = chunk["language"]
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT.format(language=LANGUAGE_NAMES[lang])},
        {"role": "user", "content": USER_PROMPT.format(text=chunk["text"])},
    ]
    started = time.time()
    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            raw, item = ask_model(messages, attempt)
            problems, details = check(item, chunk)
        except (json.JSONDecodeError, KeyError) as error:
            raw, item, problems, details = "", {}, [f"The reply was not valid JSON ({error})."], {}
        if not problems:
            break
        # Show the model its previous reply and what was wrong with it
        messages += [{"role": "assistant", "content": raw},
                     {"role": "user", "content": RETRY_PROMPT.format(problems="\n".join(f"- {p}" for p in problems))}]

    record = {
        "chunk_id": chunk["chunk_id"],
        "doc_id": chunk["doc_id"],
        "language": lang,
        "question": item.get("question", "").strip(),
        "answer": details.get("answer_span") or item.get("answer", "").strip(),
        "word_overlap": details.get("word_overlap"),
        "gold_rank": details.get("gold_rank"),
        "attempts": attempt,
        "model": MODEL,
        "temperature": TEMPERATURE,
        "seed": SEED + attempt,
    }
    if problems:
        record["problems"] = problems
        target = rejected_path
    else:
        record["question_id"] = f"q_{chunk['chunk_id']}"
        target = questions_path
    with open(target, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

    status = "ok" if not problems else "REJECTED"
    print(f"[{n}/{len(gold_chunks)}] {chunk['chunk_id']}: {status} after {attempt} attempt(s), "
          f"{time.time() - started:.0f}s  |  {record['question']}")

## 8. Summary

Check that both languages ended up with a similar number of accepted questions. A large gap
(e.g. many German chunks rejected) is itself a finding about the model, and worth noting.

In [ ]:
questions = pd.DataFrame(read_jsonl(questions_path)) if questions_path.exists() else pd.DataFrame()
rejected = pd.DataFrame(read_jsonl(rejected_path)) if rejected_path.exists() else pd.DataFrame()

summary = pd.DataFrame({
    "accepted": questions.groupby("language").size() if len(questions) else 0,
    "rejected": rejected.groupby("language").size() if len(rejected) else 0,
}).fillna(0).astype(int)
if len(questions):
    summary["mean_attempts"] = questions.groupby("language")["attempts"].mean().round(2)
    summary["mean_word_overlap"] = questions.groupby("language")["word_overlap"].mean().round(2)
summary

In [ ]:
# A few accepted questions per language, to read through
pd.set_option("display.max_colwidth", 120)
questions.groupby("language").head(5)[["language", "question", "answer", "word_overlap", "attempts"]]

In [ ]:
# Why chunks were rejected (if any)
if len(rejected):
    display(rejected[["chunk_id", "question", "answer", "problems"]])
else:
    print("No rejected chunks.")

## 9. Read the questions yourself

The checks catch most problems, but not all, for example a question that is technically answerable
but ambiguous, or an awkward German sentence. Read through `data/questions/questions.jsonl`
(or the tables above) and delete or fix any bad lines before using the questions.

To start again from scratch, delete `data/questions/questions.jsonl` and `data/questions/rejected.jsonl`
and run section 7 again.

## 10. Save to GitHub

In a terminal, from the repository folder:

```
git add notebooks/05_generate_questions.ipynb data/questions
git commit -m "Step 5: generate questions for gold chunks with Mistral"
git push
```